# AquaSense AI — Four-Parameter Training

This notebook documents the active screening profile: pH, TDS, turbidity, and temperature. Labels are generated by the project-defined screening formula, so metrics measure agreement with that formula, not laboratory potability accuracy.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report


In [ ]:
DATA = Path('../data/external/water_quality_four_parameter.csv')
df = pd.read_csv(DATA)
df.head(), df.shape, df.columns.tolist()


In [ ]:
FEATURES = ['pH', 'TDS_mg_L', 'turbidity_ntu', 'temperature_c']
required = set(FEATURES)
missing = required - set(df.columns)
if missing: raise ValueError(f'Missing columns: {missing}')
X = df[FEATURES].apply(pd.to_numeric, errors='coerce')
X = X.dropna()
print('Rows:', len(X))


## Formula-label contract

The production inference code is the source of truth. This notebook records the feature order and evaluation protocol; use `ml/train_quality_model.py` for the canonical artifact-generation command.

In [ ]:
def screening_label(row):
    ph = float(row['pH']); tds = float(row['TDS_mg_L'])
    turb = float(row['turbidity_ntu']); temp = float(row['temperature_c'])
    ph_score = 0 if 6.5 <= ph <= 8.5 else 100 * (6.5 - ph if ph < 6.5 else ph - 8.5) / 2.0
    scores = [ph_score, 100 * abs(tds) / 500, 100 * abs(turb) / 5, 100 * abs(temp - 25) / 15]
    score = min(250, sum(min(250, value) for value in scores) / 4)
    return 'Excellent' if score <= 25 else 'Good' if score <= 50 else 'Poor' if score <= 75 else 'Very Poor' if score <= 100 else 'Unsuitable'

y = X.apply(screening_label, axis=1)
y.value_counts()


In [ ]:
model = Pipeline([('imputer', SimpleImputer(strategy='median')), ('classifier', ExtraTreesClassifier(n_estimators=400, min_samples_leaf=2, random_state=42, class_weight='balanced'))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pred = cross_val_predict(model, X, y, cv=cv)
print('accuracy:', accuracy_score(y, pred))
print('macro_f1:', f1_score(y, pred, average='macro'))
print('balanced_accuracy:', balanced_accuracy_score(y, pred))
print(classification_report(y, pred, zero_division=0))


In [ ]:
model.fit(X, y)
print('Fit complete. The canonical training script also writes the joblib artifact and metadata.')
